# DQN-RL model training

<div class="alert alert-success">In this notebook, I train the Deep Q-Network Reinforcement learning model. The functions and classes from the 7_Reinforcement_learning notebook is used in this notebook. The purpose of this notebook is to train the RL model using all dataset files and evaluate the performance of the model. </div> 

In [2]:
# import all necessary libraries
import os
import time
import json
from pathlib import Path
from sklearn.utils import shuffle
import pyarrow.parquet as pq
import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Optional


## Part 1. Data

### Step 1. Retrieve the train and test files
The `load_file_names` function read and shuffle file names in pairs of train and validation before returning. 

In [11]:
def load_file_names(folder: str) -> pd.DataFrame:
    files = Path(folder).glob("*.parquet")
    filenames = []
    for file in files:
        """" File filtering """
        pf = pq.ParquetFile(f"dataset_train/{file.name}")
        # Skip the file with too few observations. 
        if pf.metadata.num_rows < 300:
            continue

        filenames.append((f"dataset_train/{file.name}", f"dataset_validation/{file.name}"))

    shuffled_file_names = shuffle(filenames, random_state=66)
    return shuffled_file_names



### Step 2. Environment State data

The `convert_data_to_state()` function loads the given file and convert the data for daily state observations. 

In [16]:
def convert_data_to_state(file_path: str, window: int = 60) -> tuple[np.ndarray, np.ndarray]:
    features = ['RSI_norm', 'MACD_line_norm', 'MACD_signal_norm', 'Volume_norm', 'Return']
    X = []
    metadata = []
    try:
        data = pd.read_parquet(file_path, engine='pyarrow')
        
        if len(data) < 300:
            return None, None

        for i in range(len(data) - window + 1): 
            X.append(data[features].iloc[i:i+window].values)
            metadata.append({
                'target': data['Target'].iloc[i+window-1], 
                'date': data.index[i+window-1], 
                'close': data['Close'].iloc[i+window-1]})
        return np.array(X), np.array(metadata)
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return None, None
    

### Step 3. Assert data integrity 

In [4]:
# loading the train and test dataset file names.
shuffled_file_names = load_file_names('dataset_train')

In [5]:
def finding_first_large_file(files: list) -> int:
    """This function returns the first larger file index"""
    for i, (train_path, _) in enumerate(files):
        try:
            if pq.read_table(train_path).num_rows > 8000:
                return i
        except Exception as e:
            print(f"Error reading {train_path}: {e}")
            return None

large_file_index = finding_first_large_file(shuffled_file_names) # Get index for the first large file
train_file, val_file = shuffled_file_names[large_file_index] if large_file_index is not None else None # selected file


<blockquote>By assertion</blockquote>

In [6]:
print("#" * 40 + " -Start- "+"#" * 40)
print("\n", chr(128994),"The ", train_file, " is selected for testing the data loading...")
state, metadata = convert_data_to_state(train_file) # read file content and convert them
assert state is not None and metadata is not None, "State conversion failed."
print("\n", chr(128992), "This file has ", state.shape[0], " days of observations.")
assert state.shape[0] == len(metadata), "Mismatch between features and metadata length."

source_data = pd.read_parquet(train_file, engine='pyarrow') # (n rows, 8 features)
obs_index = state.shape[0] - 1
source_index = source_data.shape[0] - 1 
assert source_data.iloc[source_index]['RSI_norm'] == state[obs_index][-1][0], "RSI mismatch."
assert source_data.iloc[source_index]['MACD_line_norm'] == state[obs_index][-1][1], "MACD_line mismatch."
assert source_data.iloc[source_index]['MACD_signal_norm'] == state[obs_index][-1][2], "MACD_signal mismatch."
assert source_data.iloc[source_index]['Volume_norm'] == state[obs_index][-1][3], "Volume mismatch."
assert source_data.iloc[source_index]['Return'] == state[obs_index][-1][4], "Return mismatch."
print("\n", chr(128640), "Test passed")
print("#" * 40, " -End- ","#" * 40)

######################################## -Start- ########################################

 🟢 The  dataset_train/HD.parquet  is selected for testing the data loading...

 🟠 This file has  9307  days of observations.

 🚀 Test passed
########################################  -End-  ########################################


#### Source and state data integrity check

<blockquote>By visually</blockquote>

In [7]:

indicator = "Indicator"
print("\n🍋 Source's data shape:", source_data.shape if source_data is not None else "Not Enough data available!")
print("🍀 State data shape for observations:", state.shape if state is not None else "Not Enough data available!")
print(f"\n{"Indicator":<15}|{"🍋 Source at index:" + str(source_index):<30}|{"🍀 State at index: " + str(obs_index):<30}", \
      f"\n{"RSI":<15}|{source_data.iloc[source_index]['RSI_norm']:<30}|{state[obs_index][-1][0]}", \
      f"\n{"MACD_line":<15}|{source_data.iloc[source_index]['MACD_line_norm']:<30}|{state[obs_index][-1][1]}",\
      f"\n{"MACD_signal":<15}|{source_data.iloc[source_index]['MACD_signal_norm']:<30}|{state[obs_index][-1][2]}", \
      f"\n{"Volume":<15}|{source_data.iloc[source_index]['Volume_norm']:<30}|{state[obs_index][-1][3]}", \
      f"\n{"Return":<15}|{source_data.iloc[-1]['Return']:<30}|{state[obs_index][-1][4]}")

print(f"\n{"🍋 Source value":<30}|{"🍀 Observation value":<30}", \
      f"\n{"🍊 Target: " + str(source_data.iloc[-1]['Target']):<30}|{"🍊 Target:" + str(metadata[-1]['target']):<30}")


🍋 Source's data shape: (9366, 8)
🍀 State data shape for observations: (9307, 60, 5)

Indicator      |🍋 Source at index:9365        |🍀 State at index: 9306         
RSI            |0.39044895919140316           |0.39044895919140316 
MACD_line      |1.3038538404617865            |1.3038538404617865 
MACD_signal    |1.261227551411371             |1.261227551411371 
Volume         |-0.9184159164655432           |-0.9184159164655432 
Return         |-0.0012305799601692868        |-0.0012305799601692868

🍋 Source value                |🍀 Observation value            
🍊 Target: 178.6300048828125   |🍊 Target:178.6300048828125    


In [8]:
data_to_json = [[str(path) for path in path_pair] for path_pair in shuffled_file_names]
with open("shuffled_filenames.json", "w") as f:
    json.dump(data_to_json, f)

## Part 2. DQN Environment and Agent

### Step 4. Environment

In [17]:
class MarketEnv(gym.Env):
    def __init__(self):
        # Observation space: 60 time steps × 5 features
        # Shape is (60, 5) for each observation
        self.observation_space = gym.spaces.Box(
            low=-np.inf, # lower and higher bound can be within 1 as my values are normalized. But to be safe, I will keept it to -inf. I think it won't cause any issues
            high=np.inf, 
            shape=(60, 5), 
            dtype=np.float32
        )

        # Prediction is a discrete action: 0 = hold, 1 = buy, 2 = sell
        self.action_space = gym.spaces.Discrete(3, dtype=np.int32)

        # Data and metadata will be set through load_episode_data() method
        self.data = None
        self.metadata = None
        self.current_step = 0
        self.total_score = 0.0
        self.true_positive = 0 
        self.false_positive = 0 
        self.false_negative = 0
        self.true_negative = 0 
        self.device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")
        self.terminated = False
        self.truncated = False
           
    # Helper method - translates internal state to observation (referred: Frama custom env doc)
    def _get_obs(self):
        """Convert internal state to observation format.
        self.data is (n, 60, 5) shape and self.data[self.current_step] is (60, 5) shape.
        Returns:
            A single observation of shape (60, 5) by adding a batch dimension. 
        """
        obs = self.data[self.current_step].astype(np.float32)
        # assert self.observation_space.contains(obs), f"Observation {obs.shape} does not match the observation space {self.observation_space}"
        return obs
    
    def _get_info(self):
        """Compute auxiliary information for debugging.

        Returns:
            dict: Information for the episode
        """
        return {
            "true_positive": self.true_positive,
            "false_positive": self.false_positive,
            "false_negative": self.false_negative,
            "true_negative": self.true_negative,
        }
    
    ############################################################

    def reset(self, path: str, window: int = 60, seed: Optional[int] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            path: File path to load the episode data

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)
        
        self.current_step = 0
        self.total_score = 0
        self.true_positive = 0
        self.false_positive = 0
        self.false_negative = 0
        self.true_negative = 0
        self.truncated = False
        self.terminated = False
        self.data, self.metadata = convert_data_to_state(path, window) # returns (n, 60, 5), (n, 3). In exception: None, None

        observation = self._get_obs() if self.data is not None else np.zeros((60, 5), dtype=np.float32)
        info = self._get_info() if self.metadata is not None else {"true_positive": 0, "false_positive": 0, "false_negative": 0, "true_negative": 0, "skip": True}

        return observation, info
    
    
    def step(self, action):
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-2 for hold, buy, sell)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """

        info = self._get_info()

        if self.data is None or self.metadata is None or self.current_step >= len(self.metadata):
            self.terminated = True
            self.truncated = True
            observation = np.zeros((60, 5), dtype=np.float32)
            reward = 0.0
            return observation, reward, self.terminated, self.truncated, info


        meta = self.metadata[self.current_step]
        current_close = meta['close']
        target_price = meta['target']
        

        # Safety check for invalid data.
        if target_price is None or np.isnan(target_price) or current_close is None or np.isnan(current_close):
            self.terminated = True
            self.truncated = False
            reward = 0.0
            observation = self._get_obs() if not self.terminated and not self.truncated else np.zeros((60, 5), dtype=np.float32)
            return observation, reward, self.terminated, self.truncated, info
        else:

            observation = self._get_obs() if not self.terminated and not self.truncated else np.zeros((60, 5), dtype=np.float32)

            # I don't think prices will be 0 as I just fixed the preprocessing script and generated a new dataset.
            # But just in case adding the safe skip
            if current_close == 0 or target_price == 0:
                self.terminated = False
                self.truncated = False
                reward = 0.0
                return observation, reward, self.terminated, self.truncated, info
            
            price_return = (target_price - current_close) / current_close
            
            # Determine actual trend: 0=hold, 1=buy (up), 2=sell (down)
            if price_return > 0.2:  # More than 20% increase
                actual_trend = 1  # Buy
            elif price_return < -0.2:  # More than 20% decrease
                actual_trend = 2  # Sell
            else:
                actual_trend = 0  # Hold
            
            # Reward fairly for all predictions may help agent learn better for each actions.
            if action == 0 and actual_trend == 0:
                """I assumed hold as negative sentiment."""
                reward = 2.0
                self.true_negative += 1
            elif action == 1 and actual_trend == 1:
                reward = 2.0
                self.true_positive += 1
            elif action == 2 and actual_trend == 2:
                reward = 2.0
                self.true_negative += 1
            elif action == 0 and actual_trend == 1: 
                """Missed the opportunity"""
                reward = -1.0
                self.false_negative += 1
            elif action == 0 and actual_trend == 2:
                """Couldn't avoid the loss"""
                reward = -1.0
                self.false_positive += 1
            elif action == 1 and actual_trend == 2:
                """ Harsh penalty for buy advice for a loss"""
                reward = -2.0
                self.false_positive += 1
            elif action == 1 and actual_trend == 0:
                """Wrong buy advice"""
                reward = -1.0
                self.false_positive += 1
            elif action == 2 and actual_trend == 1:
                """Sold before a rise, missed the opportunity"""
                reward = -2.0
                self.false_negative += 1
            else:
                reward = -1.0 
                self.false_negative += 1
            
            self.total_score += reward
            
            # Episode should continue unless it makes too many mistakes. This will restict it to run for full episode if it is doing very bad.
            self.terminated = self.total_score < -5000

        # Move to next step
        self.current_step += 1
        self.truncated = self.current_step >= len(self.data)
        info = self._get_info() 
        return observation, reward, self.terminated, self.truncated, info

### Step 5. Neural Network

In [18]:
class QNetwork(nn.Module):
    def __init__(self, action_space):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(5, out_channels=64, kernel_size=5),
            nn.ReLU(),
            nn.Conv1d(64, out_channels=128, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 54, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_space),
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0) 
        x = x.transpose(1, 2) # Changing the shape from (1, 60, 5) to (1, 5, 60)
        return self.network(x)

### Step 6. Replay Buffer
The replay buffer code is adapted from the "CleanRL" which itself adapted from the "Stable-Baseline3". The python module is implemented in `replaybuffer.py` file. 

In [19]:
from replaybuffer import ReplayBuffer

### Step 7. DQN Agent

In [20]:
# Copyright notice
#
# This cell contains code adapted from CleanRL DQN_atari.py
# Copyright (c) 2019 CleanRL developers

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

# Adapted the ClearRL Atari DQN implementation to fit my stock trading environment
class DQNAgent:
    def __init__(
            self, 
            env: MarketEnv,
            action_space: int = 3, 
            learning_rate: float = 1e-4,
            epsilon_decay: float = 3e-7,
            final_epsilon: float = 0.01,
            discount_factor: float = 0.99,
            buffer_size: int = 1000000,
            tau: float = 0.001):

        """
        DQN Agent.
        
        Args:
            env: MarketEnv environment
            action_space: Number of actions (hold=0, buy=1, sell=2)
            learning_rate: (Hyperparameter) Learning rate for Adam optimizer
            epsilon_decay: (Hyperparameter) Decay rate for the epsilon-greedy algorithm (exp^(-epsilon_decay * step))
            final_epsilon: (Hyperparameter) Minimum exploration rate (default 0.01 means at least 1% of the time explore)
            discount_factor: (Hyperparameter) Value of future rewards
            buffer_size: (Hyperparameter) Maximum size of the replay buffer
            tau: (Hyperparameter) Rate to update the target network at each training steps with q_network weights. (default 0.001 is closer to a full update cycle in every 1000 steps)
        """
        self.env = env
        self.action_space = action_space
        self.learning_rate = learning_rate
        self.gamma = discount_factor
        self.final_epsilon = final_epsilon
        self.epsilon_decay = epsilon_decay
        self.buffer_size = buffer_size
        self.epsilon = 1.0
        self.tau = tau
        self.device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")
        self.q_network = QNetwork(self.action_space).to(self.device)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=self.learning_rate)
        self.target_network = QNetwork(self.action_space).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        # Initializing the Replay Buffer
        self.rb = ReplayBuffer(
            self.buffer_size,
            self.env.observation_space,
            self.env.action_space,
            device=self.device,
            optimize_memory_usage=True
            )
    
    # Epsilon-greedy exploration and exploitation rate calculator
    def get_exploration_rate(self, global_step: int) -> float:
        return self.final_epsilon + (1.0 - self.final_epsilon) * np.exp(-self.epsilon_decay * global_step)
    
    def remember(self, obs, next_obs, action, reward, done):
        """
        Store experience in replay buffer.
        Input:
            obs: np.float32(60, 5) shape
            next_obs: np.float32(60, 5) shape
            action: int (0=hold, 1=buy, 2=sell)
            reward: float
            done: bool
        
        """
        reward = np.float32(reward)
        self.rb.add(obs, next_obs, action, reward, done)
    
    def act(self, observation, step: int, in_validation: bool = False):
        """
        Choose action using epsilon-greedy policy.
        Input:
            observation: np.float32(60, 5) shape
            step: step is a total processed training observations so far.
            in_validation: boolean logic to skip the epsilon-greedy
        Output:
            action: int (0=hold, 1=buy, 2=sell)
        """
        if in_validation:
            with torch.no_grad():
                observation = torch.tensor(observation, dtype=torch.float32).unsqueeze(0).to(self.device)
                q_values = self.q_network(observation)
            return int(torch.argmax(q_values, dim=1).item())
        else:
            self.epsilon = self.get_exploration_rate(step)
            if np.random.random() < self.epsilon:
                # Explore: random action
                return self.env.action_space.sample()
            else:
                with torch.no_grad():
                    # since now I am using np.float32(60, 5) shape observation, I need to change it to torch.tensor(1, 60, 5) shape before passing to the network
                    observation = torch.tensor(observation, dtype=torch.float32).unsqueeze(0).to(self.device)
                    q_values = self.q_network(observation)
                return int(torch.argmax(q_values, dim=1).item())
    
    def train(self, batch_size):
        """Train the model using random samples from buffer"""
        if self.rb.size() < batch_size:
            return
        
        # Sample random batch from replay buffer
        # Return -> ReplayBufferSamples(observations, actions, next_observations, dones, rewards) in torch tensors
        batch = self.rb.sample(batch_size)

        with torch.no_grad():
            target_max, _ = self.target_network(batch.next_observations).max(dim=1)
            td_target = batch.rewards.flatten() + self.gamma * target_max * (1 - batch.dones.flatten())
        old_val = self.q_network(batch.observations).gather(1, batch.actions).squeeze()
        # Compute loss and backpropagate
        loss = F.mse_loss(td_target, old_val)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
     
    def update_target_network(self):
        """Update target network weights."""
        for target_network_param, q_network_param in zip(self.target_network.parameters(), self.q_network.parameters()):
                    target_network_param.data.copy_(
                        self.tau * q_network_param.data + (1.0 - self.tau) * target_network_param.data
                    )
    
    
    def save_model(self, path: str):
        """Save model"""
        torch.save(self.q_network.state_dict(), path)
        print(f"model saved to {path}")
    
    def load_model(self, path: str):
        """Load model"""
        self.q_network.load_state_dict(torch.load(path))
        self.target_network.load_state_dict(self.q_network.state_dict())

### Step 8. DQN training script

In [21]:
with open("shuffled_filenames.json", "r") as f:
    file_paths = json.load(f)

In [22]:
def train_agent(
            env: MarketEnv, 
            agent: DQNAgent, 
            path_pairs: list[list[str]], 
            checkpoint_file: str = "training_state.json", 
            model_file: str = "dqn_checkpoint.pth", 
            batch_size: int = 32, 
            learning_starts: int = 80000, 
            train_frequency: int = 4):
    """
    DQN Agent training process
    
    Args:
        env: MarketEnv environment
        agent: DQN Agent instance
        path_pairs: Pair of training and validation file pathnames in a list
        checkpoint_file: The file to save last successful training step
        model_file: Model file name
        batch_size: Batch size for training
        learning_starts: Number of steps to skip the training until the buffer has enough data
        train_frequency: How often to train the model

    """
    learning_starts = learning_starts
    train_frequency = train_frequency
    checkpoint_file = checkpoint_file
    model_file = model_file
    start_index = 0
    global_step = 0
    logs = []
    
    # Load previous state if it exists
    if os.path.exists(checkpoint_file) and os.path.exists(model_file):
        with open(checkpoint_file, 'r') as f:
            current_state = json.load(f)
            start_index = current_state.get("last_index", -1) + 1
            global_step = current_state.get("global_step", 0)
            logs = current_state.get("logs", [])

        agent.load_model(model_file)
        print(f"Resuming training from {path_pairs[start_index][0]} at index {start_index}")
    
    while start_index < len(path_pairs):
        train_file, val_file = path_pairs[start_index]
        filename = os.path.basename(str(val_file))
        basename = filename.split('_processed')[0]
        try:
            #################### TRAINING ##########################
            start_time = time.time()
            print(f"Training on {train_file} (Index: {start_index})")
            obs, info = env.reset(path=train_file) 
            loss = None
            
            # If failed to read file in env.reset()
            if hasattr(info, "skip") and info["skip"]:
                logs.append({"ticker": basename, "reward": None, "precision": None, "recall": None, "train_loss": loss, "accuracy": None})
                with open(checkpoint_file, 'w') as f:
                    json.dump({"last_index": start_index, "global_step": global_step, "logs": logs }, f)
                end_time = time.time()
                print(f"Episode {start_index}: Average Reward = 0, Duration: {end_time - start_time:.2f} seconds")
                start_index += 1
                continue

            while True:
                action = agent.act(obs, global_step, in_validation=False)
                next_obs, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                agent.remember(obs, next_obs, action, reward, done)
                
                # CRUCIAL step easy to overlook
                obs = next_obs
                
                # Train on batch
                if global_step > learning_starts:
                    if global_step % train_frequency == 0:
                        loss = agent.train(batch_size)
                        # As using the soft update method, I am updating the target network at every step.
                    agent.update_target_network()
                
               
                global_step += 1
                if terminated or truncated:
                    break
    
            #################### VALIDATING ##########################
            print(f"Validating on {val_file} (Index: {start_index})")
            obs, info = env.reset(path=val_file)
            # If failed to read file in env.reset()
            if hasattr(info, "skip") and info["skip"]:
                logs.append({"ticker": basename, "reward": None, "precision": None, "recall": None, "train_loss": loss, "accuracy": None})
                with open(checkpoint_file, 'w') as f:
                    json.dump({"last_index": start_index, "global_step": global_step, "logs": logs }, f)
                end_time = time.time()
                print(f"Episode {start_index}: Average Reward = 0, Duration: {end_time - start_time:.2f} seconds")
                start_index += 1
                continue
            
            total_reward = 0.0
            steps = 0 # Figured out that I also need to normalize my metrics, otherwise my metrics are not consistent and too positive for small files. 
            while True:
                steps += 1 
                action = agent.act(obs, 15000000, in_validation=True) # giving high step number (no use, but just make sure it won't explore) and skipping the random action.
                next_obs, reward, terminated, truncated, info = env.step(action)
                done = truncated # I decided to validate until it ends no matter if it loses a lot.
                total_reward += reward
                if truncated:
                    break
            
            """Precision and recall"""
            tp = info.get("true_positive", 0)
            fp = info.get("false_positive", 0)
            fn = info.get("false_negative", 0)
            tn = info.get("true_negative", 0)

            if tp + fp > 0:
                precision = tp / (tp + fp)
            else:
                precision = None
            
            if tp + fn > 0:
                recall = tp / (tp + fn)
            else:
                recall = None

            accurate_count = tn + tp
            if steps > 0:
                accuracy = accurate_count / steps
                average_rewards = total_reward / steps 
            else:
                average_rewards = None
                accuracy = None
            
            filename = os.path.basename(str(val_file))
            basename = filename.split('_processed')[0]
            logs.append({"ticker": basename, "reward": average_rewards, "precision": precision, "recall": recall, "train_loss": loss, "accuracy": accuracy})
            end_time = time.time()
            print(f"Episode {start_index}: Average Reward = {average_rewards:.2f}, Duration: {end_time - start_time:.2f} seconds")
            start_index += 1
            # Saving the checkpoint. It will be helpful to continue in case training fails.
            agent.save_model(model_file)
            with open(checkpoint_file, 'w') as f:
                    json.dump({"last_index": start_index, "global_step": global_step, "logs": logs }, f)
        except Exception as e:
            start_index +=1 # skip the file
            print(f"Error occured in episode {train_file} at index {start_index}: {e}")


### Agent Training process

In [ ]:
env = MarketEnv()
agent = DQNAgent(env, tau=0.00025) # tau will make target-network to trail 4000 steps behind the q-network

train_agent(env, agent, file_paths)

Training on dataset_train/CAMP.parquet (Index: 0)
Validating on dataset_validation/CAMP.parquet (Index: 0)
Episode 0: Average Reward = 0.00, Duration: 23.60 seconds
model saved to dqn_checkpoint.pth


#### Error story

##### One ticker data run test result

Step 1. I selected the first file pairs for testing the DQN training
```python
temp_path = [file_paths[0]]
temp_path
```
```shell
[['train_dataset/PKE_processed.parquet',
  'validation_dataset/PKE_processed.parquet']]
```

Step 2. The training successfully completed. However I noticed too optimistic Reward point for the run. It shouldn't be right as there are different observations size (days of data), and it occured to me that I have to normalize that as well. So I implemented the normalization (by dividing the total steps for the run). 
```python
env = MarketEnv()
agent = DQNAgent(env)
train_agent(env, agent, temp_path)
```
```shell
Training on train_dataset/PKE_processed.parquet (Index: 0)</br>
Validating on validation_dataset/PKE_processed.parquet (Index: 0)</br>
Episode 0: Total Reward = 1436.00, Duration: 16.27 seconds</br>
model saved to dqn_checkpoint_1772773921.pth
```

##### An error - File with too few observations
Error occurred when there are only few observations available. 
```shell
Training on train_dataset/PHR_processed.parquet (Index: 3)
Error occured in episode train_dataset/PHR_processed.parquet at index 3: 'NoneType' object is not subscriptable
```
The file had 0 day information. I assumed it was due to the fact that the file was just listed on Nasday before April 2020. 
Corrected the env.step() method to handle None type object issue. I thought about this scenario and implemented it there in the notebook, but missed to implement the check inside the step() method.
```python
test = pd.read_parquet("train_dataset/PHR_processed.parquet", engine='pyarrow')
test.shape
```
```output
(0, 8)
```

raining on train_dataset/ASYS_processed.parquet (Index: 460)
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
/tmp/ipykernel_6738/910795320.py:112: RuntimeWarning: divide by zero encountered in scalar divide
  price_return = (target_price - current_close) / current_close

Training on train_dataset/PNRG_processed.parquet (Index: 522)
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
/tmp/ipykernel_6738/910795320.py:112: RuntimeWarning: divide by zero encountered in scalar divide
  price_return = (target_price - current_close) / current_close

<blockquote>Very promising start on my second training progress</blockquote>
My first training job took 10 hours before I stopped manually after 2600 episodes. I stopped that because I fixed many bugs for last 10 hours, and noticed the log showed no much improvement in accuracy and loss values. </br>
<img src="assets/Training_progress.png" width="600px">